# HD 189733 b Transit Photometry

**Author:** Biswajit Jana

This notebook is a small hands-on project for exploring the transit method with public TESS light-curve data.  
The default target is **HD 189733 b**, but the target settings are kept near the top so that anyone can try another known transiting planet.

The workflow is:

1. choose a target,
2. download TESS light-curve data,
3. align the data using the published transit ephemeris,
4. stack the transit windows,
5. measure the dip,
6. estimate simple physical properties such as \(R_p/R_\star\) and planet radius.

This is not a full publication-level transit fit. It is a readable starting point for learning and experimenting.

## 1. Install packages

`lightkurve` is used for downloading and handling TESS light curves.  
`pandas`, `numpy`, `matplotlib`, and `astropy` are used for analysis, plotting, and unit conversion.

In [ ]:
!pip -q install lightkurve pandas numpy matplotlib astropy

## 2. Import libraries

In [ ]:
from pathlib import Path
from urllib.parse import quote
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astropy import constants as const

import lightkurve as lk

OUTDIR = Path("outputs")
OUTDIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "font.size": 11
})

## 3. Choose a target

Change `TARGET_KEY` to try one of the preset planets.

You can also use the `CUSTOM_TARGET` block if you want to try a target that is not listed.  
For best results, choose a **known transiting planet** with a published period and transit midpoint in the NASA Exoplanet Archive.

In [ ]:
TARGET_PRESETS = {
    "HD 189733 b": {
        "host": "HD 189733",
        "planet": "HD 189733 b",
        "manual_t0_bjd": 2453988.80336,
        "reference_depth_percent": 2.41,
        "notes": "Deep hot-Jupiter transit; good demonstration target."
    },
    "HD 209458 b": {
        "host": "HD 209458",
        "planet": "HD 209458 b",
        "manual_t0_bjd": None,
        "reference_depth_percent": None,
        "notes": "Classic transiting hot Jupiter."
    },
    "WASP-12 b": {
        "host": "WASP-12",
        "planet": "WASP-12 b",
        "manual_t0_bjd": None,
        "reference_depth_percent": None,
        "notes": "Very short-period hot Jupiter; data quality can vary by sector."
    },
    "HAT-P-32 b": {
        "host": "HAT-P-32",
        "planet": "HAT-P-32 b",
        "manual_t0_bjd": None,
        "reference_depth_percent": None,
        "notes": "Inflated hot Jupiter; useful for community experiments."
    },
}

TARGET_KEY = "HD 189733 b"
CUSTOM_MODE = False

CUSTOM_TARGET = {
    "host": "TOI-700",
    "planet": "TOI-700 d",
    "manual_t0_bjd": None,
    "reference_depth_percent": None,
    "notes": "Example custom target. Change before use."
}

target_config = CUSTOM_TARGET if CUSTOM_MODE else TARGET_PRESETS[TARGET_KEY]

HOST_STAR = target_config["host"]
PLANET_NAME = target_config["planet"]
MANUAL_T0_BJD = target_config["manual_t0_bjd"]
REFERENCE_DEPTH_PERCENT = target_config["reference_depth_percent"]

MAX_PRODUCTS = 3
WINDOW_DAYS = 0.30
BIN_MINUTES = 10

print("Host star:", HOST_STAR)
print("Planet:", PLANET_NAME)
print("Notes:", target_config["notes"])

## 4. Get published planet and stellar parameters

This notebook queries the NASA Exoplanet Archive.  
The most important values are the orbital period, transit midpoint, transit duration, \(R_p/R_\star\), and stellar radius.

For custom targets, always check that these values are present and reasonable before trusting the final plot.

In [ ]:
def query_planet_from_archive(planet_name):
    base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    sql = f"""
    SELECT
        pl_name, hostname,
        pl_orbper, pl_tranmid, pl_trandur,
        pl_ratror, pl_rade, pl_radj, pl_bmassj,
        pl_orbsmax, pl_orbincl,
        st_rad, st_mass, st_teff,
        sy_dist, discoverymethod
    FROM pscomppars
    WHERE pl_name = '{planet_name}'
    """
    url = f"{base}?query={quote(sql)}&format=csv"
    return pd.read_csv(url)

archive_df = query_planet_from_archive(PLANET_NAME)

if len(archive_df) == 0:
    raise ValueError(f"No NASA Exoplanet Archive row found for {PLANET_NAME}. Check the planet name spelling.")

archive = archive_df.iloc[0]

PERIOD_DAYS = float(archive["pl_orbper"])
ARCHIVE_T0_BJD = float(archive["pl_tranmid"]) if pd.notna(archive["pl_tranmid"]) else np.nan
DURATION_HOURS = float(archive["pl_trandur"]) if pd.notna(archive["pl_trandur"]) else np.nan

RP_RS_ARCHIVE = float(archive["pl_ratror"]) if pd.notna(archive["pl_ratror"]) else np.nan
STELLAR_RADIUS_RSUN = float(archive["st_rad"]) if pd.notna(archive["st_rad"]) else np.nan
STELLAR_MASS_MSUN = float(archive["st_mass"]) if pd.notna(archive["st_mass"]) else np.nan
STELLAR_TEFF_K = float(archive["st_teff"]) if pd.notna(archive["st_teff"]) else np.nan
PLANET_MASS_MJUP = float(archive["pl_bmassj"]) if pd.notna(archive["pl_bmassj"]) else np.nan
SEMI_MAJOR_AXIS_AU = float(archive["pl_orbsmax"]) if pd.notna(archive["pl_orbsmax"]) else np.nan

if MANUAL_T0_BJD is not None:
    T0_BJD_USED = float(MANUAL_T0_BJD)
    t0_source = "manual/literature reference in target preset"
else:
    T0_BJD_USED = ARCHIVE_T0_BJD
    t0_source = "NASA Exoplanet Archive"

if not np.isfinite(PERIOD_DAYS):
    raise ValueError("Missing orbital period. This notebook needs a known transiting planet with a published period.")

if not np.isfinite(T0_BJD_USED):
    raise ValueError("Missing transit midpoint. Try another target or provide manual_t0_bjd in the target preset.")

if not np.isfinite(DURATION_HOURS):
    warnings.warn("Transit duration missing. Using 2 hours as a fallback.")
    DURATION_HOURS = 2.0

if REFERENCE_DEPTH_PERCENT is None and np.isfinite(RP_RS_ARCHIVE):
    REFERENCE_DEPTH_PERCENT = RP_RS_ARCHIVE**2 * 100

target_info = pd.DataFrame({
    "Quantity": [
        "Host star",
        "Planet",
        "Period [days]",
        "Transit midpoint used [BJD]",
        "Transit midpoint source",
        "Transit duration [hours]",
        "Archive Rp/Rs",
        "Archive/reference depth [%]",
        "Stellar radius [Rsun]",
        "Stellar mass [Msun]",
        "Stellar Teff [K]",
        "Planet mass [Mjup]",
        "Semi-major axis [AU]",
        "Distance [pc]",
        "Discovery method"
    ],
    "Value": [
        HOST_STAR,
        PLANET_NAME,
        PERIOD_DAYS,
        T0_BJD_USED,
        t0_source,
        DURATION_HOURS,
        RP_RS_ARCHIVE,
        REFERENCE_DEPTH_PERCENT,
        STELLAR_RADIUS_RSUN,
        STELLAR_MASS_MSUN,
        STELLAR_TEFF_K,
        PLANET_MASS_MJUP,
        SEMI_MAJOR_AXIS_AU,
        archive.get("sy_dist", np.nan),
        archive.get("discoverymethod", "")
    ]
})

display(target_info)
target_info.to_csv(OUTDIR / "target_info.csv", index=False)

## 5. Download TESS light curves

This is real public TESS light-curve data downloaded through Lightkurve.  
The notebook starts with only a few products so that it runs quickly in Colab.

In [ ]:
search = lk.search_lightcurve(
    HOST_STAR,
    mission="TESS",
    author="SPOC",
    exptime="short"
)

if len(search) == 0:
    warnings.warn("No SPOC short-cadence product found. Trying a broader TESS search.")
    search = lk.search_lightcurve(HOST_STAR, mission="TESS")

if len(search) == 0:
    raise RuntimeError(f"No TESS light curves found for {HOST_STAR}.")

print(search[:10])

collection = search[:MAX_PRODUCTS].download_all(quality_bitmask="default")

if collection is None or len(collection) == 0:
    raise RuntimeError("Lightkurve search succeeded, but no data downloaded.")

light_curves = []

for item in collection:
    try:
        item = item.select_flux("pdcsap_flux")
    except Exception:
        pass

    item = item.remove_nans().normalize()
    light_curves.append(item)

lc = lk.LightCurveCollection(light_curves).stitch().remove_nans().normalize()

print(lc)
print("Products used:", len(light_curves))

## 6. Look at the raw light curve

This plot is just a sanity check.  
The raw light curve may still contain stellar activity, instrumental behaviour, or sector-level offsets.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(lc.time.value, lc.flux.value, s=4, alpha=0.45, linewidths=0)
ax.set_title(f"{HOST_STAR}: TESS light curve")
ax.set_xlabel("Time [mission days]")
ax.set_ylabel("Normalised flux")

lo, hi = np.nanpercentile(lc.flux.value, [1, 99])
ax.set_ylim(lo - 0.002, hi + 0.002)

fig.tight_layout()
fig.savefig(OUTDIR / "01_tess_light_curve.png", dpi=250)
plt.show()

## 7. Predict transit times inside the observed TESS data

This is the part that uses the published ephemeris.

The flux values are real TESS data.  
The predicted transit centres are used only to align the observed data so that many individual transits can be stacked.

In [ ]:
def shift_epoch_into_observed_window(t0_bjd, observed_time, period_days):
    median_time = np.nanmedian(observed_time)

    candidates = {
        "BJD": t0_bjd,
        "BTJD": t0_bjd - 2457000.0,
        "BKJD": t0_bjd - 2454833.0,
    }

    time_system, base_t0 = min(candidates.items(), key=lambda item: abs(item[1] - median_time))
    n_shift = np.round((median_time - base_t0) / period_days)
    shifted_t0 = base_t0 + n_shift * period_days

    return shifted_t0, time_system, int(n_shift)


def make_transit_centres(time, shifted_t0, period_days):
    n_start = int(np.floor((time.min() - shifted_t0) / period_days)) - 2
    n_end = int(np.ceil((time.max() - shifted_t0) / period_days)) + 2

    centres = shifted_t0 + np.arange(n_start, n_end + 1) * period_days
    centres = centres[(centres > time.min()) & (centres < time.max())]

    return centres


time = np.asarray(lc.time.value, dtype=float)
flux = np.asarray(lc.flux.value, dtype=float)
flux = flux / np.nanmedian(flux)

good = np.isfinite(time) & np.isfinite(flux)
time = time[good]
flux = flux[good]

T0_OBS, TIME_SYSTEM, N_SHIFT = shift_epoch_into_observed_window(T0_BJD_USED, time, PERIOD_DAYS)
transit_centres = make_transit_centres(time, T0_OBS, PERIOD_DAYS)

print("Time system chosen:", TIME_SYSTEM)
print("Transit midpoint shifted into observed window:", T0_OBS)
print("Number of predicted transits:", len(transit_centres))

## 8. Stack the individual transit windows

For each predicted transit, the notebook:

1. extracts a small time window around the transit,
2. fits a local baseline using out-of-transit points,
3. divides by that baseline,
4. combines all transit windows into one folded view.

This is why the final dip becomes much clearer than a single raw transit.

In [ ]:
def extract_local_transits(time, flux, transit_centres, duration_hours, window_days=0.30):
    duration_days = duration_hours / 24.0

    all_phase_hr = []
    all_flux_norm = []
    transit_id = []

    for i, centre in enumerate(transit_centres, start=1):
        in_window = np.abs(time - centre) < window_days

        if in_window.sum() < 30:
            continue

        local_t = time[in_window] - centre
        local_f = flux[in_window]

        oot = np.abs(local_t) > 0.75 * duration_days

        if oot.sum() < 15:
            continue

        coeff = np.polyfit(local_t[oot], local_f[oot], deg=1)
        baseline = np.polyval(coeff, local_t)

        ok = np.isfinite(baseline) & (baseline > 0)

        all_phase_hr.append(local_t[ok] * 24.0)
        all_flux_norm.append(local_f[ok] / baseline[ok])
        transit_id.append(np.full(ok.sum(), i))

    if len(all_phase_hr) == 0:
        raise RuntimeError("No usable transit windows found. Try increasing MAX_PRODUCTS or WINDOW_DAYS.")

    phase_hr = np.concatenate(all_phase_hr)
    flux_norm = np.concatenate(all_flux_norm)
    transit_id = np.concatenate(transit_id)

    med = np.nanmedian(flux_norm)
    std = np.nanstd(flux_norm)
    keep = np.abs(flux_norm - med) < 5 * std

    out = pd.DataFrame({
        "phase_hr": phase_hr[keep],
        "flux_norm": flux_norm[keep],
        "transit_id": transit_id[keep]
    })

    return out


fold_df = extract_local_transits(
    time=time,
    flux=flux,
    transit_centres=transit_centres,
    duration_hours=DURATION_HOURS,
    window_days=WINDOW_DAYS
)

print("Points in stacked transit:", len(fold_df))
print("Transit windows used:", fold_df["transit_id"].nunique())

fold_df.to_csv(OUTDIR / "stacked_transit_points.csv", index=False)
display(fold_df.head())

## 9. Measure the transit depth

The basic transit-depth estimate is:

\[
\delta = 1 - \frac{F_\mathrm{in}}{F_\mathrm{out}}
\]

where \(F_\mathrm{in}\) is the median flux during transit and \(F_\mathrm{out}\) is the median out-of-transit flux.

In [ ]:
def bin_transit(fold_df, bin_minutes=10, xlim_hours=5):
    bin_width_hr = bin_minutes / 60.0
    bins = np.arange(-xlim_hours, xlim_hours + bin_width_hr, bin_width_hr)
    centres = 0.5 * (bins[:-1] + bins[1:])

    rows = []
    phase = fold_df["phase_hr"].to_numpy()
    flux = fold_df["flux_norm"].to_numpy()

    for lo, hi, centre in zip(bins[:-1], bins[1:], centres):
        m = (phase >= lo) & (phase < hi)
        if m.sum() > 2:
            rows.append({
                "phase_hr": centre,
                "flux_median": np.nanmedian(flux[m]),
                "flux_std": np.nanstd(flux[m]),
                "n_points": int(m.sum())
            })

    return pd.DataFrame(rows)


def measure_depth(fold_df, duration_hours, window_days=0.30):
    phase = fold_df["phase_hr"].to_numpy()
    flux = fold_df["flux_norm"].to_numpy()

    in_transit = np.abs(phase) < duration_hours / 2
    out_transit = (np.abs(phase) > 1.5 * duration_hours) & (np.abs(phase) < min(4.5, window_days * 24 * 0.85))

    f_in = np.nanmedian(flux[in_transit])
    f_out = np.nanmedian(flux[out_transit])

    depth_fraction = 1 - f_in / f_out

    return {
        "f_in": f_in,
        "f_out": f_out,
        "depth_fraction": depth_fraction,
        "depth_percent": depth_fraction * 100,
        "depth_ppm": depth_fraction * 1e6,
        "n_in": int(in_transit.sum()),
        "n_out": int(out_transit.sum())
    }


binned_df = bin_transit(fold_df, bin_minutes=BIN_MINUTES)
depth = measure_depth(fold_df, DURATION_HOURS, window_days=WINDOW_DAYS)

print(f"Measured depth ≈ {depth['depth_percent']:.3f}%")
print(f"Measured depth ≈ {depth['depth_ppm']:.0f} ppm")
print(f"Reference/archive depth ≈ {REFERENCE_DEPTH_PERCENT:.3f}%" if REFERENCE_DEPTH_PERCENT is not None else "No reference depth available.")

binned_df.to_csv(OUTDIR / "binned_transit.csv", index=False)

## 10. What can we estimate from the dip?

The transit depth gives the planet-to-star radius ratio:

\[
\frac{R_p}{R_\star} \approx \sqrt{\delta}
\]

If the stellar radius is known, then:

\[
R_p \approx R_\star \sqrt{\delta}
\]

This section uses the measured dip plus the archive stellar radius to estimate the planet radius.  
If a mass is available, it also estimates a rough bulk density.

In [ ]:
derived = {}

if depth["depth_fraction"] > 0:
    rp_rs_measured = np.sqrt(depth["depth_fraction"])
else:
    rp_rs_measured = np.nan

derived["Measured Rp/Rs from dip"] = rp_rs_measured
derived["Archive Rp/Rs"] = RP_RS_ARCHIVE
derived["Measured depth [%]"] = depth["depth_percent"]
derived["Reference/archive depth [%]"] = REFERENCE_DEPTH_PERCENT

if np.isfinite(rp_rs_measured) and np.isfinite(STELLAR_RADIUS_RSUN):
    rstar = STELLAR_RADIUS_RSUN * const.R_sun
    rp = rp_rs_measured * rstar

    rp_rearth = (rp / const.R_earth).decompose().value
    rp_rjup = (rp / const.R_jup).decompose().value

    derived["Estimated planet radius [R_earth]"] = rp_rearth
    derived["Estimated planet radius [R_jup]"] = rp_rjup
else:
    rp = np.nan
    derived["Estimated planet radius [R_earth]"] = np.nan
    derived["Estimated planet radius [R_jup]"] = np.nan

if np.isfinite(PLANET_MASS_MJUP) and np.isfinite(derived["Estimated planet radius [R_jup]"]):
    mass = PLANET_MASS_MJUP * const.M_jup
    radius = derived["Estimated planet radius [R_jup]"] * const.R_jup
    volume = (4/3) * np.pi * radius**3
    density = (mass / volume).to(u.g / u.cm**3)

    derived["Archive planet mass [M_jup]"] = PLANET_MASS_MJUP
    derived["Estimated bulk density [g/cm3]"] = density.value
else:
    derived["Archive planet mass [M_jup]"] = PLANET_MASS_MJUP
    derived["Estimated bulk density [g/cm3]"] = np.nan

if np.isfinite(SEMI_MAJOR_AXIS_AU):
    a = SEMI_MAJOR_AXIS_AU * u.au
    derived["Semi-major axis [AU]"] = SEMI_MAJOR_AXIS_AU
elif np.isfinite(STELLAR_MASS_MSUN):
    period = PERIOD_DAYS * u.day
    mstar = STELLAR_MASS_MSUN * const.M_sun
    a = ((const.G * mstar * period**2) / (4 * np.pi**2))**(1/3)
    a = a.to(u.au)
    derived["Semi-major axis [AU]"] = a.value
else:
    a = np.nan
    derived["Semi-major axis [AU]"] = np.nan

if np.isfinite(STELLAR_TEFF_K) and np.isfinite(STELLAR_RADIUS_RSUN) and hasattr(a, "unit"):
    rstar = STELLAR_RADIUS_RSUN * const.R_sun
    teq = STELLAR_TEFF_K * np.sqrt((rstar / (2 * a)).decompose().value)
    derived["Approx. equilibrium temperature [K]"] = teq
else:
    derived["Approx. equilibrium temperature [K]"] = np.nan

if np.isfinite(rp_rs_measured) and np.isfinite(STELLAR_RADIUS_RSUN) and hasattr(a, "unit"):
    rstar = STELLAR_RADIUS_RSUN * const.R_sun
    rp_for_prob = rp_rs_measured * rstar
    prob = ((rstar + rp_for_prob) / a).decompose().value
    derived["Approx. transit probability [%]"] = prob * 100
else:
    derived["Approx. transit probability [%]"] = np.nan

derived_df = pd.DataFrame.from_dict(derived, orient="index", columns=["value"])
display(derived_df)

derived_df.to_csv(OUTDIR / "derived_physical_parameters.csv")

## 11. Final plot

This is the main figure from the notebook.

Remember: the scatter points are real TESS measurements.  
The transit centres are predicted from the published ephemeris so that the observed transits can be aligned and stacked.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.scatter(
    fold_df["phase_hr"],
    fold_df["flux_norm"],
    s=5,
    alpha=0.20,
    linewidths=0,
    label="Folded TESS data"
)

ax.plot(
    binned_df["phase_hr"],
    binned_df["flux_median"],
    "o-",
    linewidth=2,
    markersize=4,
    label=f"{BIN_MINUTES}-min bins"
)

ax.axvline(0, linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(1, linestyle=":", linewidth=1, alpha=0.7)

ax.set_xlim(-5, 5)

low = min(0.965, np.nanpercentile(fold_df["flux_norm"], 1) - 0.003)
high = max(1.010, np.nanpercentile(fold_df["flux_norm"], 99) + 0.003)
ax.set_ylim(low, high)

title = (
    f"{PLANET_NAME}: folded TESS transit\n"
    f"Depth ≈ {depth['depth_percent']:.3f}% ({depth['depth_ppm']:.0f} ppm)"
)

if REFERENCE_DEPTH_PERCENT is not None:
    title += f", reference ≈ {REFERENCE_DEPTH_PERCENT:.3f}%"

ax.set_title(title)
ax.set_xlabel("Time from mid-transit [hours]")
ax.set_ylabel("Normalised flux")
ax.legend()

fig.tight_layout()
fig.savefig(OUTDIR / "final_transit_plot.png", dpi=300)
plt.show()

## 12. Save the results

This creates a ZIP file with the plots and CSV tables.

In [ ]:
import shutil

zip_path = shutil.make_archive("transit_photometry_results", "zip", OUTDIR)
print("Created:", zip_path)

## 13. Notes for trying other targets

Good beginner targets are usually:

- bright stars,
- short orbital periods,
- deep transits,
- known ephemerides,
- available TESS light curves.

To try another target:

1. change `TARGET_KEY`, or set `CUSTOM_MODE = True`,
2. check the archive table,
3. run the notebook from the top,
4. if no transit appears, increase `MAX_PRODUCTS`,
5. if the baseline looks poor, try changing `WINDOW_DAYS`.

For shallow planets, the dip may not be visible without more sectors or a better model.

## 14. References

Bouchy, F. et al. (2005) ‘ELODIE metallicity-biased search for transiting Hot Jupiters II. A very hot Jupiter transiting the bright K star HD189733’, *Astronomy & Astrophysics*.

Bakos, G. Á. et al. (2006) ‘Refined parameters of the planet orbiting HD 189733’, *The Astrophysical Journal*.

Pont, F. et al. (2007) ‘Hubble Space Telescope time-series photometry of the planetary transit of HD 189733: no moon, no rings, starspots’, *Astronomy & Astrophysics*.

Ricker, G. R. et al. (2015) ‘Transiting Exoplanet Survey Satellite’, *Journal of Astronomical Telescopes, Instruments, and Systems*.

Lightkurve Collaboration et al. (2018) ‘Lightkurve: Kepler and TESS time series analysis in Python’, *Astrophysics Source Code Library*.

NASA Exoplanet Archive (n.d.) ‘Planetary Systems Composite Parameters and TAP service’, NASA Exoplanet Science Institute.